#Customer Report
#####Purpose:
- This report consolidates key customer metrics and behaviors
	
#####Highlights:
1. Gathers essential fields such as names, ages, and transaction details.
2. Segments customers into categories (VIP, Regular, New) and age groups.
3. Aggregates customer-level metrics:
- total orders
- total sales
- total quantity purchased
- total products
- lifespan (in months)
4. Calculates valuable KPIs:
- recency (months since last order)
- average order value
- average monthly spend

###Create Report: gold.report_customers 

In [0]:
query = """
CREATE OR REPLACE VIEW gold.report_customers AS

-- 1) Base Query: Retrieves core columns from tables
WITH base_query AS (
    SELECT
        f.order_number,
        f.product_key,
        f.order_date,
        f.sales_amount,
        f.quantity,
        c.customer_key,
        c.customer_number,
        CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
        YEAR(CURRENT_DATE()) - YEAR(c.birthdate) AS age
    FROM gold.fact_sales AS f
    LEFT JOIN gold.dim_customers AS c
        ON c.customer_key = f.customer_key
    WHERE order_date IS NOT NULL
),
-- 2) Customer Aggregations: Summarizes key metrics at the customer level
customer_aggregation AS (
    SELECT
        customer_key,
        customer_number,
        customer_name,
        age,
        COUNT(DISTINCT order_number) AS total_orders,
        SUM(sales_amount) AS total_sales,
        SUM(quantity) AS total_quantity,
        COUNT(DISTINCT product_key) AS total_products,
        MAX(order_date) AS last_order_date,
        (YEAR(MAX(order_date)) - YEAR(MIN(order_date))) * 12 + (MONTH(MAX(order_date)) - MONTH(MIN(order_date))) AS lifespan
    FROM base_query
    GROUP BY customer_key, customer_number, customer_name, age
    ORDER BY customer_key, customer_number, customer_name, age
)
-- Final query
SELECT
    customer_key,
    customer_number,
    customer_name,
    age,
    CASE
        WHEN age < 20 THEN 'Under 20'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        ELSE '50 and above'
    END AS age_group,
    CASE
        WHEN lifespan >= 12 AND total_sales > 5000 THEN 'VIP'
        WHEN lifespan >= 12 AND total_sales <= 5000 THEN 'Regular'
        ELSE 'New'
    END AS customer_segment,
    last_order_date,
    ROUND(MONTHS_BETWEEN(CURRENT_DATE(), last_order_date), 0) AS recency,
    total_orders,
    total_sales,
    total_quantity,
    total_products,
    lifespan,
    -- Compute average order value (AVO)
    CASE
        WHEN total_orders = 0 THEN 0    -- make sure to not devide with 0
        ELSE ROUND(total_sales / total_orders, 0)
    END AS avg_order_value,
    -- Compute average monthly spend
    CASE
        WHEN lifespan = 0 THEN total_sales
        ELSE ROUND(total_sales / lifespan, 0)
    END AS avg_monthly_spend
FROM customer_aggregation
"""
df = spark.sql(query)

df.display()

print("View 'report_customers' created successfully!")


View 'report_customers' created successfully!


In [0]:
query = """
SELECT * FROM gold.report_customers
"""
df = spark.sql(query)

df.display()

customer_key,customer_number,customer_name,age,age_group,customer_segment,last_order_date,recency,total_orders,total_sales,total_quantity,total_products,lifespan,avg_order_value,avg_monthly_spend
1,AW00011000,Jon Yang,55,50 and above,VIP,2013-05-03,153.0,3,8249,8,8,28,2750.0,295.0
2,AW00011001,Eugene Huang,50,50 and above,VIP,2013-12-10,146.0,3,6384,11,10,35,2128.0,182.0
3,AW00011002,Ruben Torres,55,50 and above,VIP,2013-02-23,155.0,3,8114,4,4,25,2705.0,325.0
4,AW00011003,Christy Zhu,53,50 and above,VIP,2013-05-10,153.0,3,8139,9,9,29,2713.0,281.0
5,AW00011004,Elizabeth Johnson,47,40-49,VIP,2013-05-01,153.0,3,8196,6,6,28,2732.0,293.0
6,AW00011005,Julio Ruiz,50,50 and above,VIP,2013-05-02,153.0,3,8121,6,6,29,2707.0,280.0
7,AW00011006,Janet Alvarez,50,50 and above,VIP,2013-05-14,153.0,3,8119,5,5,28,2706.0,290.0
8,AW00011007,Marco Mehta,57,50 and above,VIP,2013-03-19,155.0,3,8211,8,8,26,2737.0,316.0
9,AW00011008,Rob Verhoff,51,50 and above,VIP,2013-03-02,155.0,3,8106,7,7,26,2702.0,312.0
10,AW00011009,Shannon Carlson,57,50 and above,VIP,2013-05-09,153.0,3,8091,5,5,28,2697.0,289.0


In [0]:
query = """
SELECT 
	age_group,
	COUNT(customer_number) AS total_customers,
	SUM(total_sales) AS total_sales
FROM gold.report_customers
GROUP BY age_group
"""
df = spark.sql(query)

df.display()

age_group,total_customers,total_sales
50 and above,12846,20686539
40-49,5636,8664719


In [0]:
query = """
SELECT 
	customer_segment,
	COUNT(customer_number) AS total_customers,
	SUM(total_sales) AS total_sales
FROM gold.report_customers
GROUP BY customer_segment
ORDER BY total_sales DESC
"""
df = spark.sql(query)

df.display()

customer_segment,total_customers,total_sales
New,14629,11086797
VIP,1653,10760470
Regular,2200,7503991
